In [3]:
import azure.ai.ml
import azure.identity

print("Azure ML SDK version:", azure.ai.ml.__version__)
print("Azure Identity imported successfully")

Azure ML SDK version: 1.34.1
Azure Identity imported successfully


In [ ]:
SUBSCRIPTION_ID = "**06e****-****-****-****-****b009****"
RESOURCE_GROUP = "rg-telecom-churn-ml"
WORKSPACE_NAME = "aml-telecom-churn"
COMPUTE_CLUSTER = "cpu-churn-cluster"

In [5]:
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()

In [6]:
from azure.ai.ml import MLClient

ml_client = MLClient(
    credential=credential,
    subscription_id=SUBSCRIPTION_ID,
    resource_group_name=RESOURCE_GROUP,
    workspace_name=WORKSPACE_NAME
)

print("MLClient created successfully")

MLClient created successfully


In [8]:
compute = ml_client.compute.get(COMPUTE_CLUSTER)

print("Compute name:", compute.name)
print("Compute type:", compute.type)
print("Provisioning state:", compute.provisioning_state)

Compute name: cpu-churn-cluster
Compute type: amlcompute
Provisioning state: Succeeded


In [8]:
for compute_resource in ml_client.compute.list():
    print(
        compute_resource.name,
        "|",
        compute_resource.type,
        "|",
        compute_resource.provisioning_state
    )

ci-churn-dev | computeinstance | Succeeded
cpu-churn-cluster | amlcompute | Succeeded


In [9]:
print("Azure ML setup validated successfully.")
print(f"Workspace: {WORKSPACE_NAME}")
print(f"Development compute: ci-churn-dev")
print(f"Training cluster: {COMPUTE_CLUSTER}")

Azure ML setup validated successfully.
Workspace: aml-telecom-churn
Development compute: ci-churn-dev
Training cluster: cpu-churn-cluster


In [10]:
from azure.ai.ml.entities import AzureBlobDatastore

print(AzureBlobDatastore)

<class 'azure.ai.ml.entities._datastore.azure_storage.AzureBlobDatastore'>


In [12]:
ml_client.datastores.get("telecom_churn_blob")

AzureBlobDatastore({'type': <DatastoreType.AZURE_BLOB: 'AzureBlob'>, 'name': 'telecom_churn_blob', 'description': 'Blob storage for Telecom Churn ML data', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/8606e131-7cc7-442a-91bc-2fd7b0096148/resourceGroups/rg-telecom-churn-ml/providers/Microsoft.MachineLearningServices/workspaces/aml-telecom-churn/datastores/telecom_churn_blob', 'Resource__source_path': '', 'base_path': '/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-churn-dev/code/Users/supulwickramasinghe7/telecom-churn-ml/notebooks', 'creation_context': None, 'credentials': {'type': 'account_key'}, 'container_name': 'churn-ml-data', 'account_name': 'sttelecomchurnml1510', 'endpoint': 'core.windows.net', 'protocol': 'https'})

In [16]:
from pathlib import Path

conda_path = Path("../environments/conda.yml").resolve()

print("Resolved path:", conda_path)
print("File exists:", conda_path.exists())

Resolved path: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-churn-dev/code/Users/supulwickramasinghe7/telecom-churn-ml/environments/conda.yml
File exists: False


In [18]:
from pathlib import Path

search_root = Path.home() / "cloudfiles" / "code" / "Users"

matches = list(
    search_root.rglob("telecom-churn-ml/environments/conda.yml")
)

for match in matches:
    print(match)

/home/azureuser/cloudfiles/code/Users/azureuser/telecom-churn-ml/environments/conda.yml


In [19]:
conda_path = Path(
    "/home/azureuser/cloudfiles/code/Users/"
    "supulwickramasinghe7/telecom-churn-ml/"
    "environments/conda.yml"
)

print("Path:", conda_path)
print("Exists:", conda_path.exists())
print("Size:", conda_path.stat().st_size if conda_path.exists() else None)

Path: /home/azureuser/cloudfiles/code/Users/supulwickramasinghe7/telecom-churn-ml/environments/conda.yml
Exists: False
Size: None


In [20]:
from pathlib import Path
from azure.ai.ml.entities import Environment

conda_path = Path(
    "/home/azureuser/cloudfiles/code/Users/"
    "azureuser/telecom-churn-ml/environments/conda.yml"
)

if not conda_path.exists():
    raise FileNotFoundError(
        f"Conda file not found: {conda_path}"
    )

training_environment = Environment(
    name="telecom-churn-training-env",
    version="1",
    description=(
        "Environment for telecom churn validation "
        "and model training."
    ),
    image=(
        "mcr.microsoft.com/azureml/"
        "openmpi4.1.0-ubuntu22.04:latest"
    ),
    conda_file=str(conda_path)
)

registered_environment = (
    ml_client.environments.create_or_update(
        training_environment
    )
)

print("Environment registered successfully")
print("Name:", registered_environment.name)
print("Version:", registered_environment.version)

Environment registered successfully
Name: telecom-churn-training-env
Version: 1


In [22]:
from pathlib import Path

src_path = Path(
    "/home/azureuser/cloudfiles/code/Users/"
    "supulwickramasinghe7/telecom-churn-ml/src"
)

validate_script = src_path / "validate.py"

print("Source folder exists:", src_path.exists())
print("validate.py exists:", validate_script.exists())

Source folder exists: True
validate.py exists: True


In [24]:
from pathlib import Path

conda_path = Path("../environments/conda.yml")

print(conda_path)
print(conda_path.exists())

../environments/conda.yml
False


In [26]:
from pathlib import Path

src_path = Path(
    "/home/azureuser/cloudfiles/code/Users/"
    "azureuser/telecom-churn-ml/src"
)

print("Source folder:", src_path.exists())
print("Validation script:", (src_path / "validate.py").exists())

Source folder: True
Validation script: True


In [27]:
from azure.ai.ml import Input, command
from azure.ai.ml.constants import InputOutputModes

validation_job = command(
    code=str(src_path),
    command=(
        "python validate.py "
        "--input-data ${{inputs.input_data}}"
    ),
    inputs={
        "input_data": Input(
            type="uri_folder",
            path="azureml:telecom_churn_features:1",
            mode=InputOutputModes.RO_MOUNT
        )
    },
    environment="azureml:telecom-churn-training-env:1",
    compute="cpu-churn-cluster",
    experiment_name="telecom-churn-data-validation",
    display_name="validate-fabric-gold-features"
)

submitted_job = ml_client.jobs.create_or_update(
    validation_job
)

print("Job name:", submitted_job.name)
print("Studio URL:", submitted_job.studio_url)

Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Uploading src (0.0 MBs): 100%|████████

Job name: sharp_pig_8qxs24mrwr
Studio URL: https://ml.azure.com/runs/sharp_pig_8qxs24mrwr?wsid=/subscriptions/8606e131-7cc7-442a-91bc-2fd7b0096148/resourcegroups/rg-telecom-churn-ml/workspaces/aml-telecom-churn&tid=7cbad65c-bf7f-438a-9d36-d3aa10965f16


In [9]:
from azure.ai.ml import load_job

training_job = load_job(
    source="../jobs/train-jobs.yaml"
)

submitted_job = ml_client.jobs.create_or_update(
    training_job
)

print("Job name:", submitted_job.name)
print("Job status:", submitted_job.status)
print("Studio URL:", submitted_job.studio_url)

Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Uploading src (0.02 MBs): 100%|███████

Job name: telecom-churn-model-training
Job status: Starting
Studio URL: https://ml.azure.com/runs/telecom-churn-model-training?wsid=/subscriptions/8606e131-7cc7-442a-91bc-2fd7b0096148/resourcegroups/rg-telecom-churn-ml/workspaces/aml-telecom-churn&tid=7cbad65c-bf7f-438a-9d36-d3aa10965f16
